In [ ]:
import os, io
from pathlib import Path
import pandas as pd
import psycopg2
import boto3
from botocore.client import Config
from dotenv import load_dotenv

_nb_file = globals().get('__vsc_ipynb_file__', '')
_project_root = Path(_nb_file).parent.parent if _nb_file else Path(__file__).parent.parent
load_dotenv(_project_root / '.env', override=True)

DB_SERVER   = os.getenv('DB_SERVER')
DB_PORT     = os.getenv('DB_PORT')
DB_USER     = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_DATABASE = os.getenv('DB_DATABASE')

MINIO_ENDPOINT   = os.getenv('MINIO_ENDPOINT')
MINIO_ACCESS_KEY = os.getenv('MINIO_ACCESS_KEY')
MINIO_SECRET_KEY = os.getenv('MINIO_SECRET_KEY')
LANDING_BUCKET   = os.getenv('MINIO_LANDING_BUCKET')

print(f'Postgres: {DB_SERVER}:{DB_PORT}/{DB_DATABASE}')
print(f'MinIO: {MINIO_ENDPOINT} | Bucket: {LANDING_BUCKET}')

In [ ]:
conn = psycopg2.connect(
    host=DB_SERVER,
    port=DB_PORT,
    user=DB_USER,
    password=DB_PASSWORD,
    dbname=DB_DATABASE
)
cursor = conn.cursor()
print(f'Conectado ao PostgreSQL [{DB_DATABASE}]')

In [ ]:
cursor.execute("""
    SELECT table_name FROM information_schema.tables
    WHERE table_type = 'BASE TABLE' AND table_schema = 'public'
    ORDER BY table_name
""")
tabelas = [row[0] for row in cursor.fetchall()]

print(f'{len(tabelas)} tabelas encontradas:')
for i, t in enumerate(tabelas, 1):
    cursor.execute(f'SELECT COUNT(*) FROM "{t}"')
    print(f'  {i:2d}. {t:<20} ({cursor.fetchone()[0]:>6} registros)')

In [ ]:
s3_client = boto3.client(
    's3',
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)

try:
    s3_client.head_bucket(Bucket=LANDING_BUCKET)
    print(f'Bucket [{LANDING_BUCKET}] já existe')
except:
    s3_client.create_bucket(Bucket=LANDING_BUCKET)
    print(f'Bucket [{LANDING_BUCKET}] criado!')

print('Buckets:', [b['Name'] for b in s3_client.list_buckets()['Buckets']])

In [ ]:
print(f'Extraindo {len(tabelas)} tabelas...\n')
resultados = []

for tabela in tabelas:
    cursor.execute(f'SELECT * FROM "{tabela}"')
    columns = [desc[0] for desc in cursor.description]
    rows = cursor.fetchall()
    df = pd.DataFrame.from_records(rows, columns=columns)

    csv_buffer = io.StringIO()
    df.to_csv(csv_buffer, index=False)
    csv_bytes = csv_buffer.getvalue().encode('utf-8')
    s3_key = f'{tabela}.csv'

    s3_client.put_object(
        Bucket=LANDING_BUCKET, Key=s3_key,
        Body=csv_bytes, ContentType='text/csv'
    )

    size_kb = len(csv_bytes) / 1024
    resultados.append({'tabela': tabela, 'registros': len(df), 'colunas': len(df.columns), 'tamanho_kb': round(size_kb, 1)})
    print(f'  {tabela}.csv -> s3://{LANDING_BUCKET}/{s3_key} ({len(df)} registros, {size_kb:.1f} KB)')

print(f'\nExtracao concluida! {len(tabelas)} tabelas exportadas.')

In [ ]:
response = s3_client.list_objects_v2(Bucket=LANDING_BUCKET)
print(f'Arquivos no bucket [{LANDING_BUCKET}]:\n')
for obj in response.get('Contents', []):
    print(f'  {obj["Key"]:<25} {obj["Size"]/1024:>8.1f} KB')

df_resumo = pd.DataFrame(resultados)
print(f'\nTotal: {df_resumo["registros"].sum():,} registros | {df_resumo["tamanho_kb"].sum():,.1f} KB')

In [ ]:
cursor.close()
conn.close()
print('Conexao PostgreSQL encerrada.')